In [1]:
import pandas as pd

df = pd.read_csv('green_tripdata_2019-10.csv.gz', compression='gzip', usecols=\
                                                                            [
                                                                            'lpep_pickup_datetime', 
                                                                            'lpep_dropoff_datetime', 
                                                                            'PULocationID', 
                                                                            'DOLocationID', 
                                                                            'passenger_count', 
                                                                            'trip_distance', 
                                                                            'tip_amount'
                                                                            ], nrows=100)
df

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount
0,2019-10-01 00:26:02,2019-10-01 00:39:58,112,196,1,5.88,0.00
1,2019-10-01 00:18:11,2019-10-01 00:22:38,43,263,1,0.80,0.00
2,2019-10-01 00:09:31,2019-10-01 00:24:47,255,228,2,7.50,0.00
3,2019-10-01 00:37:40,2019-10-01 00:41:49,181,181,1,0.90,0.00
4,2019-10-01 00:08:13,2019-10-01 00:17:56,97,188,1,2.52,2.26
...,...,...,...,...,...,...,...
95,2019-10-01 00:02:53,2019-10-01 00:14:32,126,74,1,3.10,0.00
96,2019-10-01 00:18:45,2019-10-01 00:29:23,42,74,1,1.64,0.00
97,2019-10-01 00:41:32,2019-10-01 00:52:51,75,42,1,3.17,1.50
98,2019-10-01 00:36:54,2019-10-01 00:54:20,92,179,1,5.48,0.00


In [ ]:
from dataclasses import dataclass
import dataclasses
import json

@dataclass
class TaxiRide:
    pickup_datetime: int
    dropoff_datetime: int
    pickup_location_id: int
    dropoff_location_id: int
    passenger_count: int
    trip_distance: float
    tip_amount: float

def row_to_taxi_ride(row):
    return TaxiRide(
        pickup_datetime=row['lpep_pickup_datetime'],
        dropoff_datetime=row['lpep_dropoff_datetime'],
        pickup_location_id=row['PULocationID'],
        dropoff_location_id=row['DOLocationID'],
        passenger_count=row['passenger_count'],
        trip_distance=row['trip_distance'],
        tip_amount=row['tip_amount']
    )

def trips_serializer(trip):
    trip_dict = dataclasses.asdict(trip)
    return json.dumps(trip_dict).encode('utf-8')

In [10]:
from kafka import KafkaProducer

server = 'localhost:9092'
topic_name = 'green_trips'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=trips_serializer
)

producer.bootstrap_connected()

True

In [11]:
import time

t0 = time.time()

for _, row in df.iterrows():
    trip = row_to_taxi_ride(row)
    producer.send(topic_name, value=trip)
    print("Sent trip: ", trip)
    time.sleep(0.1)
producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

TypeError: TaxiRide.__init__() got an unexpected keyword argument 'pickup_datetime'